# 02 - Experience A : decalage de domaine

On teste l'hypothese H1 : un detecteur de steganalyse entraine sur des images naturelles perd sa fiabilite sur des images generees.

Trois mesures. D'abord une AUC de reference sur le naturel. Ensuite le taux de faux positifs quand on applique ce detecteur a des images generees vierges. Enfin la chute de l'AUC en cross domaine, comparee a un detecteur reentraine sur du genere, ce qui prouve que la perte vient du domaine et non d'une impossibilite.

On charge les caracteristiques mises en cache par le notebook 01, on ne recalcule rien.

## Parametres et donnees

In [ ]:
import os, glob
import numpy as np

SEED = 42
np.random.seed(SEED)

FEATURE = 'spam'          # doit correspondre au notebook 01
ALGO = 'uniward'          # algorithme d'insertion teste, changez pour 'lsb' ou 'hill'
PAYLOAD = 0.4             # charge utile testee
GENERES = ['sd', 'sdxl', 'adm']   # sources generees a comparer au naturel

try:
    from google.colab import drive
    drive.mount('/content/drive')
    DRIVE = '/content/drive/MyDrive/memoire_data'
except Exception:
    DRIVE = '/content/memoire_data'
FEAT_DIR = f'{DRIVE}/features_{FEATURE}'
RESULTS = f'{DRIVE}/results'
os.makedirs(RESULTS, exist_ok=True)
print('Caracteristiques lues dans :', FEAT_DIR)

## 1. Chargement des caracteristiques

Pour une source donnee, on empile les covers (etiquette 0) et les porteuses (etiquette 1). Cover et porteuse sont apparies, donc alignes par ordre de fichier.

In [ ]:
def charger(source, setname):
    f = f'{FEAT_DIR}/{source}__{setname}.npy'
    if not os.path.exists(f):
        raise FileNotFoundError(f'manque : {f}. Lancez le notebook 01.')
    return np.load(f)

def jeu(source):
    # Renvoie X (covers puis porteuses) et y (0 puis 1), plus les covers seuls
    Xc = charger(source, 'cover')
    Xs = charger(source, f'{ALGO}_p{PAYLOAD}')
    n = min(len(Xc), len(Xs))
    Xc, Xs = Xc[:n], Xs[:n]
    X = np.vstack([Xc, Xs])
    y = np.concatenate([np.zeros(n), np.ones(n)])
    return X, y, Xc

X_nat, y_nat, cover_nat = jeu('natural')
print('naturel :', X_nat.shape)
for g in GENERES:
    Xg, yg, _ = jeu(g)
    print(f'{g} : {Xg.shape}')

## 2. Detecteur et validation croisee

Le detecteur est un pipeline simple : normalisation puis regression logistique. On evalue avec une validation croisee repetee, pour rapporter une AUC moyenne et son ecart type, et montrer que le resultat ne depend pas d'un decoupage chanceux.

In [ ]:
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score, RepeatedStratifiedKFold, train_test_split
from sklearn.metrics import roc_auc_score, roc_curve

def detecteur():
    return make_pipeline(StandardScaler(),
                         LogisticRegression(max_iter=5000, C=1.0))

def auc_croisee(X, y):
    # AUC moyenne et ecart type sur plusieurs decoupages
    cv = RepeatedStratifiedKFold(n_splits=5, n_repeats=3, random_state=SEED)
    s = cross_val_score(detecteur(), X, y, cv=cv, scoring='roc_auc')
    return s.mean(), s.std()

print('Detecteur pret.')

## 3. Les trois mesures

Reference sur le naturel, puis pour chaque source generee : AUC en cross domaine avec le detecteur naturel, AUC appariee avec un detecteur reentraine, et taux de faux positifs sur les covers generes.

In [ ]:
# Reference : le detecteur naturel evalue sur le naturel
auc_ref_m, auc_ref_s = auc_croisee(X_nat, y_nat)

# Detecteur naturel entraine sur tout le naturel, pour le cross domaine et les faux positifs
det_nat = detecteur().fit(X_nat, y_nat)

# Seuil calibre a environ 5 pour cent de faux positifs sur les covers naturels
score_cov_nat = det_nat.predict_proba(cover_nat)[:, 1]
seuil = np.quantile(score_cov_nat, 0.95)

res = []
for g in GENERES:
    Xg, yg, cover_g = jeu(g)
    auc_cross = roc_auc_score(yg, det_nat.predict_proba(Xg)[:, 1])
    auc_match_m, auc_match_s = auc_croisee(Xg, yg)
    fpr_g = float(np.mean(det_nat.predict_proba(cover_g)[:, 1] > seuil))
    res.append((g, auc_cross, auc_match_m, auc_match_s, fpr_g))

print(f'Reference naturel : AUC = {auc_ref_m:.3f} +/- {auc_ref_s:.3f}')
print(f'(algorithme {ALGO}, charge {PAYLOAD} bpp)\n')
print(f"{'source':8s} {'AUC cross':>10s} {'AUC apparie':>13s} {'FPR covers':>12s}")
for g, ac, am, asd, fp in res:
    print(f'{g:8s} {ac:10.3f} {am:9.3f}+/-{asd:.3f} {fp:12.3f}')
print(f'\nRappel : FPR vise sur le naturel = 0.05')

## 4. Figures

Courbes ROC et barres d'AUC, pretes pour le chapitre. On les sauvegarde sur Drive.

In [ ]:
import matplotlib.pyplot as plt

# Pour la ROC de reference, un decoupage simple du naturel
Xtr, Xte, ytr, yte = train_test_split(X_nat, y_nat, test_size=0.3, random_state=SEED, stratify=y_nat)
det_ref = detecteur().fit(Xtr, ytr)

fig, ax = plt.subplots(1, 2, figsize=(13, 5))

f, t, _ = roc_curve(yte, det_ref.predict_proba(Xte)[:, 1])
ax[0].plot(f, t, label=f'reference nat (AUC={roc_auc_score(yte, det_ref.predict_proba(Xte)[:,1]):.2f})', lw=2)
for g in GENERES:
    Xg, yg, _ = jeu(g)
    sc = det_nat.predict_proba(Xg)[:, 1]
    f, t, _ = roc_curve(yg, sc)
    ax[0].plot(f, t, label=f'cross nat vers {g} (AUC={roc_auc_score(yg, sc):.2f})')
ax[0].plot([0, 1], [0, 1], 'k--', alpha=0.4)
ax[0].set_xlabel('faux positifs'); ax[0].set_ylabel('vrais positifs')
ax[0].set_title('ROC : reference contre cross domaine'); ax[0].legend()

labels = ['ref'] + [f'{g}\ncross' for g in GENERES] + [f'{g}\napparie' for g in GENERES]
vals = [auc_ref_m] + [r[1] for r in res] + [r[2] for r in res]
couleurs = ['#2a9d8f'] + ['#e76f51'] * len(GENERES) + ['#264653'] * len(GENERES)
ax[1].bar(labels, vals, color=couleurs)
ax[1].axhline(0.5, ls='--', color='gray'); ax[1].set_ylim(0.4, 1.0)
ax[1].set_title('AUC par condition')
plt.tight_layout()
sortie = f'{RESULTS}/expA_{ALGO}_p{PAYLOAD}.png'
plt.savefig(sortie, dpi=150); plt.show()
print('Figure sauvegardee :', sortie)

## Interpretation

Le schema attendu : AUC de reference elevee, AUC cross domaine plus basse, et d'autant plus basse que la source est architecturalement eloignee du naturel. Le detecteur apparie, reentraine sur le genere, doit remonter, ce qui montre que la perte vient du changement de domaine. Enfin le taux de faux positifs sur les covers generes doit depasser nettement les 5 pour cent calibres sur le naturel.

Pensez a relancer avec `ALGO = 'lsb'` et `'hill'`, et avec `PAYLOAD = 0.2`, pour voir si le phenomene tient sur tous les cas. Notez les valeurs dans le journal, et copiez la figure dans le dossier `results` du depot.

Suite : `03_experience_B_interference` quantifie pourquoi l'insertion est plus difficile a detecter sur le genere.